# 3. Hysteresis and mixed scans: physical constraints

Run the cells in order. This demonstration uses synthetic intensities; iteration counts are illustrative, not convergence guarantees. Replace the data-generation cell with your normalized experimental intensities, support and detector mask. See [the examples guide](../docs/universal_reconstruction_examples.md).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                    if (p / "library" / "phase_retrieval_universal.py").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from library import phase_retrieval_universal as universal

# Small synthetic data make this notebook runnable without experimental files.
shape = (32, 32)
y, x = np.indices(shape)
supportmask = ((x - 16)**2 + (y - 16)**2 < 7**2).astype(float)
mask_pixel = np.zeros(shape, dtype=int)  # 0 measured; 1 invalid/unmeasured.
magnetic_pattern = np.tanh((x - 16) / 2) * supportmask

def to_field(obj):
    return np.fft.ifftshift(np.fft.ifft2(np.fft.fftshift(obj)))

def to_object(field):
    return np.fft.ifftshift(np.fft.fft2(np.fft.fftshift(field)))


In [ ]:
state_labels = ["saturated", "saturated", "domains", "domains"]
polarizations = [+1, -1, +1, -1]
energy_labels = [780.] * 4
beam_labels = ["beam"] * 4
state_maps = {"saturated": supportmask, "domains": magnetic_pattern}
objects_true = np.stack([supportmask * np.exp(-0.2 + 0.2j + p * (0.04 + 0.02j) * state_maps[s])
                         for s, p in zip(state_labels, polarizations)])
holograms = np.stack([abs(to_field(obj))**2 for obj in objects_true])


The physical model shares illumination, charge/energy response, thickness and real magnetization maps. Both helicities of a field point share a state label. For a combined energy/hysteresis scan, flatten all measurements and supply the corresponding labels. Reuse a state label across energies only for the same magnetic configuration.

In [ ]:
recipe = universal.default_universal_phase_retrieval_recipe()
recipe.update(
    modes=[1], mode_initialization="support_fft",
    warmup_mode=["HAPRE", "ER"], warmup_Nit=[40, 10],
    inner_mode=["ER"], inner_Nit=[10], outer_iterations=3,
    shuffle_observations=False,
    projection_relaxation=0.3,
    final_projection_relaxation=0,
    random_seed=0,
)
recipe.update(
    projection_model="physical_factorized",
    saturated_states={"saturated": +1}, clip_magnetization=True,
    material_mask=supportmask, material_thickness=supportmask,
    projection_constraints_inside_support_only=True,
    physical_iterations=5,
    projection_every=len(holograms),
)
# Less restrictive alternative: projection_model="state_energy_beam".
# It fits additive log-object components, not bounded magnetization maps.
# Other optional controls: fit_material_thickness, physical_phase_reference,
# charge_spectral_constraint and magnetic_spectral_constraint (see the guide).
fields, warmup, components, bsmasks, errors = universal.universal_phase_retrieval_algorithm(
    holograms, mask_pixel, supportmask,
    state_labels=state_labels, energy_labels=energy_labels,
    polarization_coefficients=polarizations, illumination_labels=beam_labels,
    universal_recipe=recipe,
)
magnetization = components["magnetization_by_state"]
fig, axes = plt.subplots(1, len(magnetization), figsize=(8, 3))
for ax, (state, image) in zip(axes, magnetization.items()):
    ax.imshow(image, vmin=-1, vmax=1, cmap="RdBu_r"); ax.set_title(state)
plt.tight_layout(); plt.show()
print("Physical residual:", components["fit_residual_rms"])


A saturation anchor fixes magnetic scale, not absolute thickness/optical constants. Keep enough shared metadata to identify the components. A final detector projection can disturb physical consistency: inspect both detector and physical residuals. Synthetic examples demonstrate API usage, not guaranteed recovery of the ground truth.